In [ ]:
"""
How to use:
1. Put transactions in input_data.txt (one per line).
2. Format: <step>,<type>,<amount>,<nameOrig>,<oldbalanceOrg>,<newbalanceOrig>,<nameDest>,<oldbalanceDest>,<newbalanceDest>,<isFlaggedFraud>
3. Run this script.
4. Open output_analysis.txt to see predictions + Phi‑3 explanations.

Note: this takes about 2-4 minutes per transaction
"""

# Install dependencies if needed
%pip install torch --index-url https://download.pytorch.org/whl/cpu
%pip install transformers
%pip install accelerate
%pip install pandas joblib numpy


# Imports
import json
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Fairness: group thresholds from training
group_thresholds = None
fairness_metrics = {}

# Load Phi‑3
model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map=None
).to("cpu")

def run_phi3(prompt, max_new_tokens=60, temperature=0.7):
    chat_prompt = f"<|user|>\n{prompt}\n<|assistant|>"
    inputs = tokenizer(chat_prompt, return_tensors="pt").to("cpu")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        eos_token_id=tokenizer.eos_token_id
    )
    
    # Prevents the model from hallucinating new prompts and continuing the interaction on its own
    parts = tokenizer.decode(outputs[0], skip_special_tokens=True).split("You are ")
    '''
    print(f"Tokenizer: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")
    '''
    return (f"You are {parts[1]}")

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [52]:
# Load classifier with fairness support
model_path = Path.cwd() / "../code/custom-classifier-model/fraud_detector_forest.pkl"
model_path = model_path.resolve()
artifact = joblib.load(model_path)

if isinstance(artifact, dict) and "pipeline" in artifact:
    pipeline = artifact["pipeline"]
    threshold = float(artifact.get("threshold", 0.5))
    feature_columns = artifact.get("feature_columns", [])
    group_thresholds = artifact.get("group_thresholds", None)  # Fairness thresholds
    fairness_metrics = artifact.get("fairness_metrics", {})
else:
    pipeline = artifact
    threshold = 0.5
    feature_columns = []
    group_thresholds = None
    fairness_metrics = {}

if feature_columns:
    feature_columns = list(feature_columns)

# Load preprocessing artifacts used during training
preprocessing_path = Path.cwd() / "../code/data-prep/preprocessing_artifacts.pkl"
preprocessing_path = preprocessing_path.resolve()
preprocessing_artifacts = joblib.load(preprocessing_path)
scaler = preprocessing_artifacts["scaler"]
cat_encoder = preprocessing_artifacts["cat_encoder"]
numeric_features = preprocessing_artifacts["numeric_features"]
categorical_features = preprocessing_artifacts["categorical_features"]
processed_feature_columns = numeric_features + categorical_features

print(f"Loaded fraud model from: {model_path}")
print(f"Threshold = {threshold}")
print(f"Feature columns = {feature_columns}")
print(f"Numeric features = {numeric_features}")
print(f"Categorical features = {categorical_features}")
if group_thresholds:
    print(f"Group thresholds (fairness): {group_thresholds}")
if fairness_metrics:
    print(f"Fairness metrics: {fairness_metrics}")
print(f"Loaded preprocessing artifacts from: {preprocessing_path}")

# Load regression model
reg_model_path = Path.cwd() / "../code/custom-regression-model/fraud_detector_regression.pkl"
regressor = joblib.load(reg_model_path)
print(f"Loaded regression model from: {reg_model_path}")

Loaded fraud model from: C:\Users\mbaoj\CSCE581-Spring2025-MatthewBojanowski\code\custom-classifier-model\fraud_detector_forest.pkl
Threshold = 0.5245807839433445
Feature columns = ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'nameOrig_prefix', 'nameDest_prefix', 'orig_balance_delta', 'dest_balance_delta', 'amount_to_oldOrg', 'amount_to_newOrg']
Numeric features = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'orig_balance_delta', 'dest_balance_delta', 'amount_to_oldOrg', 'amount_to_newOrg']
Categorical features = ['type', 'nameOrig_prefix', 'nameDest_prefix']
Group thresholds (fairness): {np.int64(0): np.float64(0.1), np.int64(1): np.float64(0.1)}
Fairness metrics: {'test_fpr_gap': 4.7211331558886695e-05, 'test_fnr_gap': 0.0040650406504065045, 'test_accuracy': 0.9999476106802963}
Loaded preprocessing artifacts from: C:\Users\mbaoj\CSCE581-Spring2025-MatthewBojanowski\code\data-prep\preproc

In [53]:
# Feature engineering
def prepare_features(df):
    df = df.copy()
    df['type'] = df['type'].astype(str)
    df['nameOrig'] = df['nameOrig'].astype(str)
    df['nameDest'] = df['nameDest'].astype(str)

    df['nameOrig_prefix'] = df['nameOrig'].str[0]
    df['nameDest_prefix'] = df['nameDest'].str[0]
    df['orig_balance_delta'] = df['newbalanceOrig'] - df['oldbalanceOrg']
    df['dest_balance_delta'] = df['newbalanceDest'] - df['oldbalanceDest']

    old_org_denom = df['oldbalanceOrg'].replace(0, np.nan).fillna(1)
    new_org_denom = df['newbalanceOrig'].replace(0, np.nan).fillna(1)
    df['amount_to_oldOrg'] = df['amount'] / old_org_denom
    df['amount_to_newOrg'] = df['amount'] / new_org_denom

    # Apply the saved preprocessing artifacts from training.
    numeric_df = df[numeric_features].astype(float).fillna(0)
    categorical_df = df[categorical_features].astype(str).fillna('UNKNOWN')
    categorical_encoded = cat_encoder.transform(categorical_df)
    numeric_scaled = scaler.transform(numeric_df)

    processed = np.concatenate([numeric_scaled, categorical_encoded], axis=1)
    return pd.DataFrame(processed, columns=feature_columns)


In [54]:
# Phi‑3 LLM analysis
def get_llm_explanation(transaction, prediction, probability, threshold):
    fraud_status = "FRAUD" if prediction == 1 else "NOT FRAUD"

    prompt = f"""
You are a financial fraud analyst explaining a transaction classification.

A machine learning model classified a transaction as {fraud_status}.

TRANSACTION DETAILS:
- Transaction type: {transaction.get('type')}
- Amount: ${transaction.get('amount'):,.2f}
- Originator old balance: ${transaction.get('oldbalanceOrg'):,.2f}
- Originator new balance: ${transaction.get('newbalanceOrig'):,.2f}
- Destination old balance: ${transaction.get('oldbalanceDest'):,.2f}
- Destination new balance: ${transaction.get('newbalanceDest'):,.2f}
- Flagged by system: {"Yes" if transaction.get('isFlaggedFraud') == 1 else "No"}

MODEL OUTPUT:
- Fraud probability: {probability:.4f} (threshold: {threshold:.4f})
- Classification: {fraud_status}

TASK:
Write a clear, 2–3 sentence explanation of why the model made this classification.
Focus on the key risk factors that influenced the decision.
"""
    return run_phi3(prompt, max_new_tokens=180, temperature=0.4)

# Phi‑3 LLM analysis for regression
def get_llm_explanation_regression(transaction, predicted_amount):
    prompt = f"""
You are a financial analyst explaining a transaction amount prediction.

TRANSACTION DETAILS:
- Transaction type: {transaction.get('type')}
- Amount: ${transaction.get('amount'):,.2f}
- Originator old balance: ${transaction.get('oldbalanceOrg'):,.2f}
- Originator new balance: ${transaction.get('newbalanceOrig'):,.2f}
- Destination old balance: ${transaction.get('oldbalanceDest'):,.2f}
- Destination new balance: ${transaction.get('newbalanceDest'):,.2f}
- Flagged by system: {"Yes" if transaction.get('isFlaggedFraud') == 1 else "No"}

TASK:
Write a clear, 2–3 sentence explanation of why the model predicted this amount.
Focus on the key factors that influenced the prediction.
"""

    return run_phi3(prompt, max_new_tokens=180, temperature=0.4)

# Analyze data for fraud with fairness-aware predictions
def analyze_data(user_input_data, use_fair_threshold=True):
    if isinstance(user_input_data, dict):
        user_df = pd.DataFrame([user_input_data])
    else:
        user_df = pd.DataFrame(user_input_data)

    expected_columns = [
        'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg',
        'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'
    ]
    user_df = user_df[expected_columns]

    features = prepare_features(user_df)
    probabilities = pipeline.predict_proba(features)[:, 1]
    reg_predictions = regressor.predict(features)
    
    # Apply fairness-aware threshold if available
    if use_fair_threshold and group_thresholds is not None:
        # Use group-specific thresholds based on isFlaggedFraud
        predictions = np.zeros(len(probabilities), dtype=int)
        for idx, row in user_df.iterrows():
            flag = row.get('isFlaggedFraud', 0)
            group_thresh = group_thresholds.get(flag, threshold)
            predictions[idx] = 1 if probabilities[idx] >= group_thresh else 0
        used_threshold = f"group-specific: {group_thresholds}"
    else:
        predictions = (probabilities >= threshold).astype(int)
        used_threshold = threshold

    results = []
    for idx, row in user_df.iterrows():
        prob = float(probabilities[idx])
        pred = int(predictions[idx])
        
        # Get the threshold used for this prediction
        if use_fair_threshold and group_thresholds is not None:
            flag = row.get('isFlaggedFraud', 0)
            pred_threshold = group_thresholds.get(flag, threshold)
        else:
            pred_threshold = threshold
            
        llm_text = get_llm_explanation(row.to_dict(), pred, prob, pred_threshold)
        llm_text_reg = get_llm_explanation_regression(row.to_dict(), float(reg_predictions[idx]))
        conclusion = 'Fraud detected' if pred == 1 else 'No fraud detected'

        results.append({
            **row.to_dict(),
            'prediction': pred,
            'fraud_probability': prob,
            'threshold_used': pred_threshold,
            'predicted_amount': float(reg_predictions[idx]),
            'llm_explanation': llm_text,
            'llm_explanation_regression': llm_text_reg,
        })

    return pd.DataFrame(results)

In [55]:
# File input and output for data and results
def load_input_data(filepath):
    columns = [
        'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg',
        'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'
    ]
    return pd.read_csv(filepath, header=None, names=columns)

def write_output_analysis(result_df, output_path):
    result_df = result_df.copy()
    csv_text = result_df.drop(columns=['llm_explanation', 'llm_explanation_regression']).to_csv(index=False)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(csv_text)
        f.write("\n\nLLM Explanations (Classifier):\n")
        for idx, row in result_df.iterrows():
            f.write(f"Line {idx+1}: {row['llm_explanation']}\n\n")
        f.write("\n\nLLM Explanations (Regressor):\n")
        for idx, row in result_df.iterrows():
            f.write(f"Line {idx+1}: {row['llm_explanation_regression']}\n\n")

    print(f"Wrote analysis to {output_path}")

# Run full pipeline with fairness
input_filepath = "input_data.txt"
output_filepath = "output_analysis.txt"
input_data = load_input_data(input_filepath)

# Use fair threshold for predictions
results = analyze_data(input_data, use_fair_threshold=True)
write_output_analysis(results, output_filepath)

# Note: running this takes about 2-4 minutes per transaction
results.head()

Wrote analysis to output_analysis.txt


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFlaggedFraud,prediction,fraud_probability,threshold_used,predicted_amount,llm_explanation,llm_explanation_regression
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,1,0.109811,0.1,-0.230469,You are a financial fraud analyst explaining a...,You are a financial analyst explaining a trans...
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,1,0.144785,0.1,-0.281693,You are a financial fraud analyst explaining a...,You are a financial analyst explaining a trans...
2,1,TRANSFER,182.00,C1305486145,182.0,0.00,C553264065,0.0,0.0,0,1,0.174066,0.1,-0.288962,You are a financial fraud analyst explaining a...,You are a financial analyst explaining a trans...
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,0,1,0.179053,0.1,-0.288962,You are a financial fraud analyst explaining a...,You are a financial analyst explaining a trans...
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,1,0.162260,0.1,-0.273713,You are a financial fraud analyst explaining a...,You are a financial analyst explaining a trans...
